In [ ]:
# Cargar eda_wl_clean y configurar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

eda_wl_clean = pd.read_csv('../data/eda_wl_clean.csv')
# Convertir columnas de fecha a datetime por si hicieran falta
for col in ['date', 'w_dob', 'l_dob']:
    if col in eda_wl_clean.columns:
        eda_wl_clean[col] = pd.to_datetime(eda_wl_clean[col], errors='coerce')

print("eda_wl_clean cargado exitosamente.")

# Preguntas



## Qué luchador tiene mayor probabilidad de ganar una pelea?

Respondo utilizando Random Forest para clasificación

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix, roc_curve, auc

# ==========================================
# 1. Preparación de variables y Manejo de NaNs
# ==========================================
w_cols = ['w_height', 'w_reach', 'w_str_acc', 'w_td_acc', 'w_str_def', 'w_td_def', 'w_sapm', 'w_splm']
l_cols = ['l_height', 'l_reach', 'l_str_acc', 'l_td_acc', 'l_str_def', 'l_td_def', 'l_sapm', 'l_splm']
features = ['height', 'reach', 'str_acc', 'td_acc', 'str_def', 'td_def', 'sapm', 'splm']
target = 'is_winner'

df_w = eda_wl_clean[w_cols].copy()
df_w.columns = features
df_w['is_winner'] = 1

df_l = eda_wl_clean[l_cols].copy()
df_l.columns = features
df_l['is_winner'] = 0

df_q1 = pd.concat([df_w, df_l], ignore_index=True)
df_q1 = df_q1.dropna(subset=['is_winner'])
X = df_q1[features]
y = df_q1['is_winner']

# Imputación de NaNs en las variables predictoras usando la mediana (robusto a outliers)
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)

# Separación en conjunto de entrenamiento y test
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y, test_size=0.2, random_state=42)

# ==========================================
# 2. Entrenamiento del Modelo (Random Forest)
# ==========================================
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)

# ==========================================
# 3. Función predictiva de probabilidad
# ==========================================
def predict_win_probability(fighter_stats, model, imp):
    """
    Recibe un DataFrame de 1 fila con las métricas de un luchador,
    aplica la misma imputación del entrenamiento y devuelve la probabilidad de victoria.
    """
    stats_imputed = imp.transform(fighter_stats)
    # predict_proba devuelve [Prob_Clase0, Prob_Clase1]
    proba_victoria = model.predict_proba(stats_imputed)[0][1] 
    return proba_victoria

# Ejemplo de uso de la función (descomentar para probar):
# stats_luchador_ejemplo = pd.DataFrame([[180, 185, 45, 30, 55, 60, 3.5, 4.0]], columns=features)
# print("Probabilidad de ganar:", predict_win_probability(stats_luchador_ejemplo, rf_model, imputer))

# ==========================================
# 4. Evaluación del Modelo (Matriz y Curva ROC)
# ==========================================
y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)[:, 1]

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de Confusión
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax[0])
ax[0].set_title('Matriz de Confusión', fontsize=14)
ax[0].set_xlabel('Predicción (0=Pierde, 1=Gana)')
ax[0].set_ylabel('Realidad')

# Curva ROC y AUC
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)
ax[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'Curva ROC (AUC = {roc_auc:.2f})')
ax[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
ax[1].set_title('Curva ROC', fontsize=14)
ax[1].set_xlabel('Tasa de Falsos Positivos (FPR)')
ax[1].set_ylabel('Tasa de Verdaderos Positivos (TPR)')
ax[1].legend(loc="lower right")

plt.tight_layout()
plt.show()


NameError: name 'eda_wl_clean' is not defined

## Evaluación extendida y Baseline
Se presenta el modelo base (baseline), un Árbol de Decisión, y las métricas detalladas por clase tanto en Train como en Test (matriz de confusión, precision, recall, f1).

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import time

# Baseline
baseline_rojo = 711
baseline_azul = 506
total_baseline = baseline_rojo + baseline_azul
baseline_acc = baseline_rojo / total_baseline
print(f"Baseline: {baseline_rojo} victorias del rojo, {baseline_azul} del azul => {baseline_acc:.2%} de predecir siempre rojo.")

# Entrenar Árbol de Decisión
dt_model = DecisionTreeClassifier(criterion='entropy', max_depth=5, random_state=42, class_weight='balanced')
start_time = time.time()
dt_model.fit(X_train, y_train)
dt_time = time.time() - start_time
print(f"Tiempo de entrenamiento (Árbol de Decisión): {dt_time:.4f} segundos")

y_train_pred = dt_model.predict(X_train)
y_test_pred = dt_model.predict(X_test)

tree_acc = accuracy_score(y_test, y_test_pred)
mejora_pp = (tree_acc - baseline_acc) * 100
print(f"Precisión del árbol en test: {tree_acc:.2%}")
print(f"Mejora con respecto al baseline: {mejora_pp:.2f} puntos porcentuales (pp).")

print("\n--- Métricas en Train ---")
print(classification_report(y_train, y_train_pred, target_names=['Gana Azul (0)', 'Gana Rojo (1)']))
print("\nMatriz de Confusión (Train):")
print(confusion_matrix(y_train, y_train_pred))

print("\n--- Métricas en Test ---")
print(classification_report(y_test, y_test_pred, target_names=['Gana Azul (0)', 'Gana Rojo (1)']))
print("\nMatriz de Confusión (Test):")
print(confusion_matrix(y_test, y_test_pred))
